
# From Global Equivalence to Local Recovery
## Single-notebook reproduction of manuscript figures and numerical analysis

This notebook reproduces the numerical post-processing and Figures 1--7 for:

**Othman H. Y. Zalloum, _From Global Equivalence to Local Recovery: Energy-Recovery Spectra and Residual-Controlled Halo Corrections for Adaptive Finite Elements_.**

It uses the public reproducibility archive at GitHub release `v1.0.0`. In Google Colab, run all cells from top to bottom. The notebook writes publication-ready PDF and PNG figures to `manuscript_outputs/` and creates a ZIP at the end.

The numerical solvers are not rerun here; this notebook reproduces the **analysis and plotting** from the archived CSV outputs used by the manuscript.


In [ ]:

from pathlib import Path
import os, subprocess, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# In Colab, the notebook clones the frozen public release v1.0.0.
# For an offline/local verification run, set AFEM_DATA_ROOT to a directory containing the CSV files.
local_data = os.environ.get('AFEM_DATA_ROOT')
if local_data:
    DATA = Path(local_data)
else:
    REPO = Path('/content/adaptive-residual-correction-fem')
    if not REPO.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'v1.0.0',
            'https://github.com/ozalloum/adaptive-residual-correction-fem.git',
            str(REPO)
        ], check=True)
    DATA = REPO / 'data'

OUT = Path(os.environ.get('AFEM_OUTPUT_DIR', '/content/manuscript_outputs'))
OUT.mkdir(parents=True, exist_ok=True)
PLOT_CSV = OUT / 'plot_csv'
PLOT_CSV.mkdir(parents=True, exist_ok=True)
print('Data directory:', DATA)
print('Output directory:', OUT)
print('Plot CSV directory:', PLOT_CSV)


In [ ]:

# Load the archived result tables.
files = {
    'uniform': 'uniform.csv',
    'afem': 'afem.csv',
    'locality': 'locality_spectrum.csv',
    'localized': 'localized_correction.csv',
    'slopes': 'convergence_slopes.csv',
    'dorfler': 'dorfler_sensitivity.csv',
    'interface': 'interface_estimator_ablation.csv',
    'rchc': 'rchc_tau_sensitivity.csv',
    'timing': 'timing_final_25reps.csv',
    'verification': 'verification_checks.csv',
}
D = {k: pd.read_csv(DATA / v) for k, v in files.items()}
for k, df in D.items():
    print(f'{k:12s}: {len(df):5d} rows')


In [ ]:

# Plotting conventions used consistently across the manuscript.
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'grid.linestyle': '-',
})

problem_order = [
    'Smooth Poisson',
    'L-shaped singularity',
    'High-contrast interface (1e4)',
    'Localized reaction-diffusion layer',
]
problem_display = {
    'Smooth Poisson': 'Smooth Poisson',
    'L-shaped singularity': 'L-shaped singularity',
    'High-contrast interface (1e4)': r'High-contrast interface $10^4$',
    'Localized reaction-diffusion layer': 'Localized reaction-diffusion layer',
}
problem_styles = {
    'Smooth Poisson': dict(color='tab:blue', marker='o', linestyle='-'),
    'L-shaped singularity': dict(color='tab:red', marker='s', linestyle='--'),
    'High-contrast interface (1e4)': dict(color='tab:green', marker='^', linestyle='-.'),
    'Localized reaction-diffusion layer': dict(color='tab:purple', marker='D', linestyle=':'),
}

def panel_label(ax, label, x=-0.13, y=1.04):
    """Put a panel label outside the plotting area."""
    ax.text(x, y, label, transform=ax.transAxes, ha='left', va='top',
            fontsize=14, fontweight='bold', clip_on=False)

def expand_log_limits(values, frac=0.10):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    lo, hi = np.log10(values.min()), np.log10(values.max())
    pad = frac * (hi - lo if hi > lo else 1.0)
    return 10**(lo - pad), 10**(hi + pad)

def save_figure(fig, stem):
    fig.savefig(OUT / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUT / f'{stem}.png', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close(fig)


## Figures 1 and 2: uniform versus adaptive convergence

In [ ]:

uniform, afem = D['uniform'], D['afem']

for metric, stem, ylabel in [
    ('Energy', 'Fig01_energy_convergence', 'Energy error'),
    ('L2', 'Fig02_l2_convergence', r'$L^2$ error'),
]:
    fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.2), constrained_layout=True)
    for ax, prob, lab in zip(axes.flat, problem_order, ['(a)', '(b)', '(c)', '(d)']):
        du = uniform[uniform['Problem'] == prob].sort_values('DOF')
        da = afem[afem['Problem'] == prob].sort_values('DOF')
        ax.loglog(du['DOF'], du[metric], color='tab:blue', marker='o', linestyle='--',
                  label=r'Uniform $P_1$', linewidth=1.6, markersize=5)
        ax.loglog(da['DOF'], da[metric], color='tab:orange', marker='s', linestyle='-',
                  label='AFEM-direct', linewidth=1.6, markersize=5)
        ax.set_xlabel('Degrees of freedom')
        ax.set_ylabel(ylabel)
        ax.legend(loc='upper right', frameon=False)
        ax.grid(True, which='both', alpha=0.25)
        panel_label(ax, lab)
    save_figure(fig, stem)

# Machine-readable data used in Figures 1 and 2.
fig01_data = pd.concat([
    uniform[['Problem','Method','DOF','Energy']].copy(),
    afem[['Problem','Method','DOF','Energy']].copy(),
], ignore_index=True)
fig01_data.to_csv(PLOT_CSV / 'Fig01_energy_convergence.csv', index=False)

fig02_data = pd.concat([
    uniform[['Problem','Method','DOF','L2']].copy(),
    afem[['Problem','Method','DOF','L2']].copy(),
], ignore_index=True)
fig02_data.to_csv(PLOT_CSV / 'Fig02_l2_convergence.csv', index=False)


### Convergence-slope verification

In [ ]:

# Independently recompute the stored log(error)-versus-log(DOF) slopes over the archived fitting ranges.
def recompute_slope(problem, method, metric, dof_min, dof_max):
    src = uniform if method == 'Uniform' else afem
    method_name = 'Uniform' if method == 'Uniform' else 'AFEM-direct'
    d = src[(src['Problem'] == problem) & (src['Method'] == method_name)]
    d = d[(d['DOF'] >= dof_min) & (d['DOF'] <= dof_max)].sort_values('DOF')
    return np.polyfit(np.log(d['DOF'].to_numpy()), np.log(d[metric].to_numpy()), 1)[0]

slope_check = D['slopes'].copy()
slope_check['RecomputedSlope'] = [
    recompute_slope(r.Problem, r.Method, r.Metric, r.DOF_min, r.DOF_max)
    for r in slope_check.itertuples()
]
slope_check['AbsDifference'] = (slope_check['RecomputedSlope'] - slope_check['Slope']).abs()
display(slope_check)
print('Maximum slope discrepancy:', slope_check['AbsDifference'].max())


### Matched-DOF comparisons quoted in the manuscript

In [ ]:

def nearest_uniform(problem, target_dof):
    d = uniform[uniform['Problem'] == problem].copy()
    return d.iloc[(d['DOF'] - target_dof).abs().argsort().iloc[0]]

matched_rows = []
for prob in ['L-shaped singularity', 'Smooth Poisson']:
    a = afem[afem['Problem'] == prob].sort_values('Level').iloc[-1]
    u = nearest_uniform(prob, a['DOF'])
    matched_rows.append({
        'Problem': prob,
        'AFEM_DOF': int(a['DOF']), 'Uniform_DOF': int(u['DOF']),
        'AFEM_Energy': a['Energy'], 'Uniform_Energy': u['Energy'],
        'Energy_improvement_factor': u['Energy']/a['Energy'],
        'AFEM_L2': a['L2'], 'Uniform_L2': u['L2'],
        'L2_improvement_factor': u['L2']/a['L2'],
    })
display(pd.DataFrame(matched_rows))


## Figures 3 and 4: energy-recovery locality and work-normalized recovery

In [ ]:

loc = D['locality']
parts = []
for prob in problem_order:
    dp = loc[loc['Problem'] == prob]
    finest = dp['Level'].max()
    parts.append(dp[dp['Level'] == finest].copy())
locf = pd.concat(parts, ignore_index=True)

fig, ax = plt.subplots(figsize=(9.2, 6.3), constrained_layout=True)
for prob in problem_order:
    d = locf[locf['Problem'] == prob].sort_values('ActiveFraction')
    s = problem_styles[prob]
    ax.plot(d['ActiveFraction'], d['RecoveryFraction'], label=problem_display[prob],
            color=s['color'], marker=s['marker'], linestyle=s['linestyle'],
            linewidth=1.8, markersize=7)
ax.axhline(0.99, color='0.45', linestyle=':', linewidth=1.4)
ax.set_xlim(0.38, 1.02)
ax.set_ylim(0.90, 1.005)
ax.set_xlabel(r'Active interior-DOF fraction $\phi(W)$')
ax.set_ylabel(r'Energy-recovery fraction $\rho(W)$')
ax.legend(loc='lower right', frameon=False)
save_figure(fig, 'Fig03_locality_spectrum')

fig, ax = plt.subplots(figsize=(9.2, 6.3), constrained_layout=True)
for prob in problem_order:
    d = locf[locf['Problem'] == prob].sort_values('ActiveFraction')
    s = problem_styles[prob]
    ax.plot(d['ActiveFraction'], d['WorkNormalizedRecovery'], label=problem_display[prob],
            color=s['color'], marker=s['marker'], linestyle=s['linestyle'],
            linewidth=1.8, markersize=7)
ax.axhline(1.0, color='0.45', linestyle=':', linewidth=1.4)
ax.set_xlim(0.40, 1.02)
ax.set_ylim(0.96, 2.22)
ax.set_xlabel(r'Active interior-DOF fraction $\phi(W)$')
ax.set_ylabel(r'Work-normalized recovery $J(W)=\rho/\phi$')
ax.legend(loc='upper right', frameon=False)
save_figure(fig, 'Fig04_work_normalized_recovery')

# Machine-readable data used in Figures 3 and 4 (finest recorded level per benchmark).
locf[['Problem','Level','Halo','DOF','InteriorDOF','ActiveDOF','ActiveFraction',
      'Energy','DirectEnergy','ProlongedEnergy','RecoveryFraction',
      'WorkNormalizedRecovery']].to_csv(PLOT_CSV / 'Fig03_locality_spectrum.csv', index=False)
locf[['Problem','Level','Halo','ActiveFraction','RecoveryFraction',
      'WorkNormalizedRecovery']].to_csv(PLOT_CSV / 'Fig04_work_normalized_recovery.csv', index=False)


### Finest-level locality summary

In [ ]:

locality_summary = locf[locf['Halo'].isin([0, 1])][
    ['Problem','Halo','ActiveFraction','RecoveryFraction','WorkNormalizedRecovery']
].sort_values(['Problem','Halo'])
display(locality_summary)
one_halo = locf[locf['Halo'] == 1]
print('Minimum one-halo recovery:', one_halo['RecoveryFraction'].min())
print('Maximum one-halo active fraction:', one_halo['ActiveFraction'].max())


## Figure 5: Dörfler-marking sensitivity

In [ ]:

sens = D['dorfler']
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.6), constrained_layout=True)
colors = ['tab:red', 'tab:green', 'tab:purple']
for ax, prob, color, lab in zip(axes, sens['Problem'].unique(), colors, ['(a)', '(b)', '(c)']):
    d = sens[sens['Problem'] == prob].sort_values('DOF')
    ax.loglog(d['DOF'], d['Energy'], color=color, marker='o', linestyle='-',
              linewidth=1.8, markersize=7)
    for _, r in d.iterrows():
        ax.annotate(rf'$\theta$={r["Theta"]:.1f}', (r['DOF'], r['Energy']),
                    xytext=(6, 6), textcoords='offset points', fontsize=10)
    ax.set_xlabel('DOF')
    ax.set_ylabel('Energy error')
    # Deliberately larger horizontal margin so the theta=0.7 annotation stays inside each panel.
    ax.set_xlim(*expand_log_limits(d['DOF'], frac=0.32))
    ax.set_ylim(*expand_log_limits(d['Energy'], frac=0.10))
    ax.grid(True, which='both', alpha=0.25)
    panel_label(ax, lab, x=-0.17, y=1.03)
save_figure(fig, 'Fig05_dorfler_sensitivity')

display(sens[['Problem','Theta','DOF','Energy','Estimator','Cumulative_s']])

# Machine-readable data used in Figure 5.
sens[['Problem','Theta','Level','DOF','Energy','Estimator','Cumulative_s','MinAngle']].to_csv(PLOT_CSV / 'Fig05_dorfler_sensitivity.csv', index=False)


## Coefficient-weighting ablation for the high-contrast interface

In [ ]:

interface = D['interface']
final_interface = interface.sort_values('Level').groupby('Problem', as_index=False).tail(1)
display(final_interface[['Problem','EstimatorType','DOF','L2','Energy','Cumulative_s']])

weighted = final_interface[final_interface['Problem'] == 'High-contrast interface (1e4)'].iloc[0]
unweighted = final_interface[final_interface['Problem'] == 'Interface-unweighted'].iloc[0]
interface_changes = pd.DataFrame({
    'Metric': ['DOF','Energy','L2','Cumulative_s'],
    'Weighted': [weighted.DOF, weighted.Energy, weighted.L2, weighted.Cumulative_s],
    'Unweighted': [unweighted.DOF, unweighted.Energy, unweighted.L2, unweighted.Cumulative_s],
})
interface_changes['Percent_change_weighted_vs_unweighted'] = 100*(interface_changes['Weighted']/interface_changes['Unweighted'] - 1)
display(interface_changes)


## Figure 6: residual-controlled halo selection

In [ ]:

# The manuscript Figure 6 reports averages over the last three recorded levels for each benchmark.
rchc = D['rchc']
avg_parts = []
for prob in problem_order:
    dp = rchc[rchc['Problem'] == prob]
    last3 = sorted(dp['Level'].unique())[-3:]
    q = (dp[dp['Level'].isin(last3)]
         .groupby(['Problem','Tau'], as_index=False)
         .agg(EnergyRatio=('EnergyRatio','mean'), ActiveFraction=('ActiveFraction','mean')))
    avg_parts.append(q)
ravg = pd.concat(avg_parts, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 5.3))
for ax, ycol, ylabel, lab in [
    (axes[0], 'EnergyRatio', 'Localized/direct energy-error ratio', '(a)'),
    (axes[1], 'ActiveFraction', 'Active interior-DOF fraction', '(b)'),
]:
    for prob in problem_order:
        d = ravg[ravg['Problem'] == prob].sort_values('Tau')
        s = problem_styles[prob]
        ax.plot(d['Tau'], d[ycol], label=problem_display[prob],
                color=s['color'], marker=s['marker'], linestyle=s['linestyle'],
                linewidth=1.8, markersize=6)
    if ycol == 'EnergyRatio':
        ax.axhline(1.0, color='0.45', linestyle=':', linewidth=1.4)
    ax.set_xlabel(r'Residual threshold $\tau$')
    ax.set_ylabel(ylabel)
    panel_label(ax, lab, x=-0.12, y=1.03)
    ax.grid(True, which='both', alpha=0.25)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.005))
fig.subplots_adjust(bottom=0.20, left=0.08, right=0.98, top=0.96, wspace=0.18)
save_figure(fig, 'Fig06_rchc_sensitivity')

display(ravg)

# Machine-readable averaged data plotted in Figure 6.
ravg[['Problem','Tau','EnergyRatio','ActiveFraction']].to_csv(PLOT_CSV / 'Fig06_rchc_sensitivity.csv', index=False)


## Figure 7: repeated localized-workflow tradeoff

In [ ]:

localized = D['localized']
# Final recorded state of each direct AFEM benchmark.
direct_final = afem.sort_values('Level').groupby('Problem', as_index=False).tail(1).set_index('Problem')
# Final recorded state of each localized method/benchmark.
local_final = localized.sort_values('Level').groupby(['Problem','Method'], as_index=False).tail(1).copy()
local_final['EnergyRatio'] = [r.Energy / direct_final.loc[r.Problem, 'Energy'] for r in local_final.itertuples()]
local_final['TimeRatio'] = [r.Cumulative_s / direct_final.loc[r.Problem, 'Cumulative_s'] for r in local_final.itertuples()]
local_final['L2Ratio'] = [r.L2 / direct_final.loc[r.Problem, 'L2'] for r in local_final.itertuples()]

method_markers = {'HLRC-r0':'o', 'HLRC-r1':'s', 'HLRC-r2':'^', 'RCHC':'D'}
fig, ax = plt.subplots(figsize=(9.2, 6.3), constrained_layout=True)
for prob in problem_order:
    d = local_final[local_final['Problem'] == prob]
    color = problem_styles[prob]['color']
    for _, r in d.iterrows():
        ax.scatter(r['TimeRatio'], r['EnergyRatio'], s=80, color=color,
                   marker=method_markers[r['Method']], zorder=3)
        ax.annotate(r['Method'], (r['TimeRatio'], r['EnergyRatio']), xytext=(4, 4),
                    textcoords='offset points', fontsize=9)
ax.axvline(1.0, color='0.45', linestyle=':', linewidth=1.4)
ax.axhline(1.0, color='0.45', linestyle=':', linewidth=1.4)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.80, 2.45)
ax.set_ylim(0.95, 3.35)
ax.set_xlabel('Cumulative-time ratio to AFEM-direct')
ax.set_ylabel('Energy-error ratio to AFEM-direct')
legend_handles = [
    Line2D([0],[0], marker='o', linestyle='None', markersize=8,
           markerfacecolor=problem_styles[p]['color'], markeredgecolor=problem_styles[p]['color'],
           label=problem_display[p]) for p in problem_order
]
ax.legend(handles=legend_handles, loc='upper right', frameon=False)
save_figure(fig, 'Fig07_workflow_tradeoff')

display(local_final[['Problem','Method','EnergyRatio','L2Ratio','TimeRatio','ActiveFraction']].sort_values(['Problem','Method']))

# Machine-readable final-state ratios plotted in Figure 7.
local_final[['Problem','Method','Level','DOF','ActiveDOF','ActiveFraction','Energy','L2','Cumulative_s','EnergyRatio','L2Ratio','TimeRatio']].sort_values(['Problem','Method']).to_csv(PLOT_CSV / 'Fig07_workflow_tradeoff.csv', index=False)


## Dedicated 25-repeat timing audit

In [ ]:

timing = D['timing']
# Build the compact manuscript timing table from the archived medians.
timing_rows = []
for prob in problem_order:
    d = timing[timing['Problem'] == prob]
    direct = d[d['Method'] == 'Direct assembly+solve'].iloc[0]
    row = {
        'Problem': prob,
        'DirectMedian_s': direct.Median_s,
        'Q25_s': direct.Q25_s,
        'Q75_s': direct.Q75_s,
    }
    for method, key in [
        ('HLRC-r0 local assembly+solve','r0/direct'),
        ('HLRC-r1 local assembly+solve','r1/direct'),
        ('HLRC-r2 local assembly+solve','r2/direct'),
    ]:
        r = d[d['Method'] == method].iloc[0]
        row[key] = r.Median_s / direct.Median_s
    timing_rows.append(row)
timing_table = pd.DataFrame(timing_rows)
display(timing_table)


## Verification summary

In [ ]:

verification = D['verification']
display(verification)
print(f"Passing checks: {verification['Pass'].sum()} / {len(verification)}")
assert verification['Pass'].all(), 'At least one archived verification check failed.'


## Package the reproduced outputs

In [ ]:
zip_path = OUT.parent / 'AFEM_Manuscript_Figures_Analysis_Output.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():
            z.write(f, arcname=f.relative_to(OUT))
print('Created:', zip_path)
print('Generated files:')
for f in sorted(OUT.rglob('*')):
    if f.is_file():
        print(' -', f.relative_to(OUT))
